In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import csv, os, re, time, sys

# ============== CONFIG ==============
USER = "HN744"
PWD  = "Imecolcase2026##"

LOGIN_URL = "https://portal.cnh.com/DPLogin/Login.do?operation=login"
CSPS_URL  = "https://my.dlrportal.com/myservices/external-links/displaylink?appName=CSPS&userId=HN744"

INPUT_CSV  = "partes_entrada.csv"     # <-- tu archivo de entrada
OUTPUT_CSV = "precios_y_stock.csv"    # <-- archivo de salida
DELAY_BETWEEN_QUERIES = 0.8           # segundos para no saturar

# ============ SELENIUM SETUP ============
opts = Options()
opts.add_argument("--start-maximized")
# opts.add_argument("--headless=new")   # opcional
driver = webdriver.Chrome(options=opts)
W = WebDriverWait(driver, 30)

# =============== HELPERS ===============
def tnum(s: str) -> str:
    s = (s or "").replace("\xa0", " ").strip()
    if not any(ch.isdigit() for ch in s): return ""
    return s.replace(" ", "").replace(".", "").replace(",", ".")

def tint(s: str) -> int:
    s = re.sub(r"[^\d]", "", s or "")
    return int(s) if s else 0

def tx(xpath: str, wait=30) -> str:
    el = WebDriverWait(driver, wait).until(EC.presence_of_element_located((By.XPATH, xpath)))
    return el.text.strip()

def sum_disponibilidad() -> int:
    # 3 reintentos por refrescos del DOM
    xp_cells = "//table[.//th[contains(., 'Disponibilidad')]]/tbody//tr[td and count(td)>=2]/td[2]"
    for _ in range(3):
        try:
            cells = driver.find_elements(By.XPATH, xp_cells)
            return sum(tint(c.get_attribute("textContent")) for c in cells)
        except Exception:
            time.sleep(0.3)
    cells = driver.find_elements(By.XPATH, xp_cells)
    return sum(tint(c.get_attribute("textContent")) for c in cells)

def ensure_in_detail_frame():
    """Asegura que estamos dentro del frame 'detail'."""
    driver.switch_to.default_content()
    WebDriverWait(driver, 25).until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "detail")))

def open_consulta_general():
    """Abre 'Consulta General de Referencia' desde el menú (solo si es necesario)."""
    driver.switch_to.default_content()
    WebDriverWait(driver, 25).until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "menu")))
    # Carpeta (icono) y luego el link
    driver.find_element(By.ID, "folderIcon9").click()
    driver.find_element(By.XPATH, "//a[@title='Consulta General de Referencia']").click()
    ensure_in_detail_frame()

def login_and_open_csps():
    # 1) Login
    driver.get(LOGIN_URL)
    W.until(EC.presence_of_element_located((By.ID, "userID"))).send_keys(USER)
    driver.find_element(By.ID, "password").send_keys(PWD)
    driver.find_element(By.ID, "btn-submit").click()

    # 2) Esperar portal y abrir CSPS en nueva pestaña
    WebDriverWait(driver, 40).until(EC.url_contains("my.dlrportal.com"))
    driver.switch_to.new_window("tab")
    driver.get(CSPS_URL)
    WebDriverWait(driver, 40).until(EC.presence_of_element_located((By.TAG_NAME, "frameset")))

    # 3) Menú -> Consulta General de Referencia
    open_consulta_general()

def query_part(part_number: str):
    """Consulta una referencia, devuelve dict con datos o levanta excepción si falla."""
    ensure_in_detail_frame()

    # Campo de referencia y botón
    part_input = W.until(EC.presence_of_element_located((By.ID, "partNr")))
    part_input.clear()
    part_input.send_keys(part_number)
    driver.find_element(By.ID, "displayBtn").click()

    # Esperar bloque de precios
    WebDriverWait(driver, 40).until(
        EC.presence_of_element_located((By.XPATH, "//td[contains(., 'Máquina Parada Order')]"))
    )

    # Extraer campos
    ref_    = tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'Referencia')]]/td[2]")
    pn_mp   = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'Precio Neto')]]/td[2]"))
    pn_norm = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'Precio Neto')]]/td[3]"))
    pl_mp   = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'PriceList')]]/td[2]"))
    pl_norm = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'PriceList')]]/td[3]"))

    # Sumar disponibilidad (si existe el bloque)
    stock_total = ""
    try:
        hdr = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//th[contains(., 'Disponibilidad')]"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", hdr)
        time.sleep(0.2)
        stock_total = sum_disponibilidad()
    except TimeoutException:
        stock_total = ""  # No había bloque de disponibilidad
    except Exception:
        stock_total = ""  # Algún fallo puntual: dejamos vacío

    return {
        "Entrada": part_number,
        "Referencia": ref_,
        "PrecioNeto_MaqParada": pn_mp,
        "PrecioNeto_Normal":   pn_norm,
        "PriceList_MaqParada": pl_mp,
        "PriceList_Normal":    pl_norm,
        "StockTotal":          stock_total
    }

def read_parts_from_csv(path: str):
    """Lee part numbers desde CSV. Usa columna 'part_number' si existe; si no, toma la primera."""
    parts = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        rows = list(reader)
        if not rows:
            return parts
        # Detectar encabezado
        header = [h.strip().lower() for h in rows[0]]
        start_idx = 1
        col_idx = 0
        if any(h for h in header):
            # Tiene encabezado
            if "part_number" in header:
                col_idx = header.index("part_number")
            else:
                col_idx = 0  # primera columna
        else:
            # Sin encabezado
            start_idx = 0
            col_idx = 0

        for r in rows[start_idx:]:
            if not r: 
                continue
            val = (r[col_idx] or "").strip()
            if val:
                parts.append(val)
    return parts

def append_output_row(path, row_dict, write_header_if_needed=True):
    write_header = write_header_if_needed and (not os.path.isfile(path) or os.path.getsize(path) == 0)
    with open(path, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow([
                "Entrada",
                "Referencia",
                "PrecioNeto_MaqParada", "PrecioNeto_Normal",
                "PriceList_MaqParada",  "PriceList_Normal",
                "StockTotal",
                "Status", "Mensaje"
            ])
        w.writerow([
            row_dict.get("Entrada", ""),
            row_dict.get("Referencia", ""),
            row_dict.get("PrecioNeto_MaqParada", ""),
            row_dict.get("PrecioNeto_Normal", ""),
            row_dict.get("PriceList_MaqParada", ""),
            row_dict.get("PriceList_Normal", ""),
            row_dict.get("StockTotal", ""),
            row_dict.get("Status", ""),
            row_dict.get("Mensaje", "")
        ])

# =============== MAIN FLOW ===============
try:
    # Login y navegación inicial
    login_and_open_csps()

    # Leer partes
    parts = read_parts_from_csv(INPUT_CSV)
    if not parts:
        print(f"[ERROR] El CSV '{INPUT_CSV}' no tiene datos.")
        sys.exit(1)
    print(f"[INFO] Partes a consultar: {len(parts)}")

    # Procesar una a una
    for i, p in enumerate(parts, 1):
        # Reintentos por referencia para errores puntuales
        max_attempts = 2
        last_err = None
        for attempt in range(1, max_attempts + 1):
            try:
                data = query_part(p)
                data["Status"]  = "OK"
                data["Mensaje"] = ""
                append_output_row(OUTPUT_CSV, data)
                print(f"[{i}/{len(parts)}] OK -> {p}  Ref={data.get('Referencia','')}  Stock={data.get('StockTotal','')}")
                break
            except TimeoutException as te:
                # Si el frame/detail cambió o se cerró, reabre el módulo y reintenta
                last_err = te
                try:
                    open_consulta_general()
                except Exception:
                    pass
                time.sleep(0.5)
            except Exception as e:
                last_err = e
                time.sleep(0.5)
        else:
            # Si no logró en reintentos, registramos error
            append_output_row(OUTPUT_CSV, {
                "Entrada": p,
                "Referencia": "",
                "PrecioNeto_MaqParada": "",
                "PrecioNeto_Normal": "",
                "PriceList_MaqParada": "",
                "PriceList_Normal": "",
                "StockTotal": "",
                "Status": "ERROR",
                "Mensaje": str(last_err)[:250]
            })
            print(f"[{i}/{len(parts)}] ERROR -> {p}: {last_err}")

        time.sleep(DELAY_BETWEEN_QUERIES)

    print(f"\n[FIN] CSV actualizado: {OUTPUT_CSV}")

except Exception as e:
    print("[FATAL]", e)

finally:
    try:
        input("ENTER para cerrar...")
    except:
        pass
    driver.quit()



[INFO] Partes a consultar: 16
[1/16] OK -> 9992069  Ref=9992069  Stock=10742
[2/16] OK -> 403145A1  Ref=403145A1  Stock=0
[3/16] OK -> 4802127  Ref=4802127  Stock=198
[4/16] OK -> 87573794  Ref=87573794  Stock=249
[5/16] OK -> 504088112  Ref=504088112  Stock=77
[6/16] OK -> 4600753  Ref=4600753  Stock=1334
[7/16] OK -> 4031577  Ref=4031577  Stock=805
[8/16] OK -> 47909212  Ref=47909212  Stock=95
[9/16] OK -> 47909214  Ref=47909214  Stock=115
[10/16] OK -> 14464881  Ref=14464881  Stock=0
[11/16] OK -> 99439562  Ref=99439562  Stock=471
[12/16] OK -> 98418138  Ref=98418138  Stock=1143
[13/16] OK -> 98424046  Ref=98424046  Stock=266
[14/16] OK -> 84478262  Ref=84478262  Stock=0
[15/16] ERROR -> 500378759: Message: element click intercepted: Element <input type="submit" name="submitDispatchAction" value="Consultar" onclick="onDisplayBtn()" class="submit" id="displayBtn"> is not clickable at point (444, 127). Other element would receive the click: <td width="148" class="dark">...</td>
  (Ses

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
)
import csv, os, re, time, sys

# ============== CONFIG ==============
USER = "HN744"
PWD  = "Imecolcase2026##"

LOGIN_URL = "https://portal.cnh.com/DPLogin/Login.do?operation=login"
CSPS_URL  = "https://my.dlrportal.com/myservices/external-links/displaylink?appName=CSPS&userId=HN744"

INPUT_CSV  = "partes_entrada.csv"     # Archivo de entrada con part numbers (columna 'part_number' o 1a columna)
OUTPUT_CSV = "precios_y_stock.csv"    # Archivo de salida
DELAY_BETWEEN_QUERIES = 0.8           # segundos

# ============ SELENIUM SETUP ============
opts = Options()
opts.add_argument("--start-maximized")
# opts.add_argument("--headless=new")   # opcional
driver = webdriver.Chrome(options=opts)
W = WebDriverWait(driver, 30)

# =============== HELPERS ===============
def tnum(s: str) -> str:
    s = (s or "").replace("\xa0", " ").strip()
    if not any(ch.isdigit() for ch in s): return ""
    return s.replace(" ", "").replace(".", "").replace(",", ".")

def tint(s: str) -> int:
    s = re.sub(r"[^\d]", "", s or "")
    return int(s) if s else 0

def tx(xpath: str, wait=30) -> str:
    el = WebDriverWait(driver, wait).until(EC.presence_of_element_located((By.XPATH, xpath)))
    return el.text.strip()

def sum_disponibilidad() -> int:
    xp_cells = "//table[.//th[contains(., 'Disponibilidad')]]/tbody//tr[td and count(td)>=2]/td[2]"
    for _ in range(3):
        try:
            cells = driver.find_elements(By.XPATH, xp_cells)
            return sum(tint(c.get_attribute("textContent")) for c in cells)
        except Exception:
            time.sleep(0.3)
    cells = driver.find_elements(By.XPATH, xp_cells)
    return sum(tint(c.get_attribute("textContent")) for c in cells)

def ensure_in_detail_frame():
    driver.switch_to.default_content()
    WebDriverWait(driver, 25).until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "detail")))

def open_consulta_general():
    driver.switch_to.default_content()
    WebDriverWait(driver, 25).until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "menu")))
    driver.find_element(By.ID, "folderIcon9").click()
    driver.find_element(By.XPATH, "//a[@title='Consulta General de Referencia']").click()
    ensure_in_detail_frame()

def login_and_open_csps():
    # 1) Login
    driver.get(LOGIN_URL)
    W.until(EC.presence_of_element_located((By.ID, "userID"))).send_keys(USER)
    driver.find_element(By.ID, "password").send_keys(PWD)
    driver.find_element(By.ID, "btn-submit").click()

    # 2) Esperar portal y abrir CSPS en nueva pestaña
    WebDriverWait(driver, 40).until(EC.url_contains("my.dlrportal.com"))
    driver.switch_to.new_window("tab")
    driver.get(CSPS_URL)
    WebDriverWait(driver, 40).until(EC.presence_of_element_located((By.TAG_NAME, "frameset")))

    # 3) Menú -> Consulta General de Referencia
    open_consulta_general()

def trigger_consulta_with_enter_on_error(part_input):
    """
    Intenta el click normal. Si hay error (interceptado o no-clickable por timeout),
    envía ENTER sobre el input como fallback.
    """
    try:
        btn = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "displayBtn")))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        btn.click()
        return
    except (ElementClickInterceptedException, TimeoutException):
        # Fallback: ENTER sobre el input
        try:
            part_input.click()
        except Exception:
            pass
        try:
            # blur opcional para asegurar procesamiento del DOM
            driver.execute_script("arguments[0].blur()", part_input)
        except Exception:
            pass
        part_input.send_keys(Keys.ENTER)

def query_part(part_number: str):
    ensure_in_detail_frame()

    # Campo de referencia y botón
    part = W.until(EC.presence_of_element_located((By.ID, "partNr")))
    part.clear()
    part.send_keys(part_number)
    trigger_consulta_with_enter_on_error(part)

    # Esperar bloque de precios
    WebDriverWait(driver, 40).until(
        EC.presence_of_element_located((By.XPATH, "//td[contains(., 'Máquina Parada Order')]"))
    )

    # Extraer campos
    ref_    = tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'Referencia')]]/td[2]")
    pn_mp   = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'Precio Neto')]]/td[2]"))
    pn_norm = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'Precio Neto')]]/td[3]"))
    pl_mp   = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'PriceList')]]/td[2]"))
    pl_norm = tnum(tx("//tbody[.//td[contains(., 'Máquina Parada Order')]]//tr[td[1][contains(., 'PriceList')]]/td[3]"))

    # Sumar disponibilidad
    stock_total = ""
    try:
        hdr = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//th[contains(., 'Disponibilidad')]"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", hdr)
        time.sleep(0.2)
        stock_total = sum_disponibilidad()
    except TimeoutException:
        stock_total = ""
    except Exception:
        stock_total = ""

    return {
        "Entrada": part_number,
        "Referencia": ref_,
        "PrecioNeto_MaqParada": pn_mp,
        "PrecioNeto_Normal":   pn_norm,
        "PriceList_MaqParada": pl_mp,
        "PriceList_Normal":    pl_norm,
        "StockTotal":          stock_total
    }

def read_parts_from_csv(path: str):
    parts = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        rows = list(reader)
        if not rows:
            return parts
        header = [h.strip().lower() for h in rows[0]]
        start_idx = 1
        col_idx = 0
        if any(h for h in header):
            if "part_number" in header:
                col_idx = header.index("part_number")
            else:
                col_idx = 0
        else:
            start_idx = 0
            col_idx = 0

        for r in rows[start_idx:]:
            if not r:
                continue
            val = (r[col_idx] or "").strip()
            if val:
                parts.append(val)
    return parts

def append_output_row(path, row_dict, write_header_if_needed=True):
    write_header = write_header_if_needed and (not os.path.isfile(path) or os.path.getsize(path) == 0)
    with open(path, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow([
                "Entrada",
                "Referencia",
                "PrecioNeto_MaqParada", "PrecioNeto_Normal",
                "PriceList_MaqParada",  "PriceList_Normal",
                "StockTotal",
                "Status", "Mensaje"
            ])
        w.writerow([
            row_dict.get("Entrada", ""),
            row_dict.get("Referencia", ""),
            row_dict.get("PrecioNeto_MaqParada", ""),
            row_dict.get("PrecioNeto_Normal", ""),
            row_dict.get("PriceList_MaqParada", ""),
            row_dict.get("PriceList_Normal", ""),
            row_dict.get("StockTotal", ""),
            row_dict.get("Status", ""),
            row_dict.get("Mensaje", "")
        ])

# =============== MAIN FLOW ===============
try:
    login_and_open_csps()

    parts = read_parts_from_csv(INPUT_CSV)
    if not parts:
        print(f"[ERROR] El CSV '{INPUT_CSV}' no tiene datos.")
        sys.exit(1)
    print(f"[INFO] Partes a consultar: {len(parts)}")

    for i, p in enumerate(parts, 1):
        max_attempts = 2
        last_err = None
        for attempt in range(1, max_attempts + 1):
            try:
                data = query_part(p)
                data["Status"]  = "OK"
                data["Mensaje"] = ""
                append_output_row(OUTPUT_CSV, data)
                print(f"[{i}/{len(parts)}] OK -> {p}  Ref={data.get('Referencia','')}  Stock={data.get('StockTotal','')}")
                break
            except (TimeoutException, StaleElementReferenceException) as te:
                last_err = te
                try:
                    open_consulta_general()
                except Exception:
                    pass
                time.sleep(0.5)
            except Exception as e:
                last_err = e
                time.sleep(0.5)
        else:
            append_output_row(OUTPUT_CSV, {
                "Entrada": p,
                "Referencia": "",
                "PrecioNeto_MaqParada": "",
                "PrecioNeto_Normal": "",
                "PriceList_MaqParada": "",
                "PriceList_Normal": "",
                "StockTotal": "",
                "Status": "ERROR",
                "Mensaje": str(last_err)[:250]
            })
            print(f"[{i}/{len(parts)}] ERROR -> {p}: {last_err}")

        time.sleep(DELAY_BETWEEN_QUERIES)

    print(f"\n[FIN] CSV actualizado: {OUTPUT_CSV}")

except Exception as e:
    print("[FATAL]", e)

finally:
    try:
        input("ENTER para cerrar...")
    except:
        pass
    driver.quit()


[INFO] Partes a consultar: 88
[1/88] OK -> 5088004  Ref=5088004  Stock=44
[2/88] OK -> 5088003  Ref=5088003  Stock=48
[3/88] OK -> 5088002  Ref=5088002  Stock=49
[4/88] OK -> 98475474  Ref=98475474  Stock=37
[5/88] OK -> 17284481  Ref=17284481  Stock=201
[6/88] OK -> 99445564  Ref=99445564  Stock=71
[7/88] OK -> 47747210  Ref=47747210  Stock=35
[8/88] OK -> 1907837  Ref=1907837  Stock=211
[9/88] OK -> 5097521  Ref=5097521  Stock=0
[10/88] OK -> 4802127  Ref=4802127  Stock=198
[11/88] OK -> 87573794  Ref=87573794  Stock=249
[12/88] OK -> 504088112  Ref=504088112  Stock=77
[13/88] OK -> 4600753  Ref=4600753  Stock=1334
[14/88] OK -> 4031577  Ref=4031577  Stock=805
[15/88] OK -> 47909212  Ref=47909212  Stock=95
[16/88] OK -> 47909214  Ref=47909214  Stock=115
[17/88] OK -> 14464881  Ref=14464881  Stock=0
[18/88] OK -> 99439562  Ref=99439562  Stock=469
[19/88] OK -> 98418138  Ref=98418138  Stock=1137
[20/88] OK -> 98424046  Ref=98424046  Stock=266
[21/88] OK -> 84478262  Ref=84478262  Stock